# Clean pasted text and count it

Clean text copied from any journal article and count paragraphs, words and characters.

1. Paste your text into **Original text** and click **Find line-end hyphens**.
2. Select any splits that should keep a genuine hyphen, then click **Apply cleanup**.
3. Review and edit **Cleaned text**. Paragraph, word and character counts update as you type.
4. Use **Copy paragraph count** and **Copy word count** to copy plain digits without dots, commas or labels.

The default paragraph rule counts body lines ending in `.`, `:`, `?` or `!`, including trailing footnote numbers directly attached to the punctuation (for example, `.12` or `?3`). Numbers starting with zero are excluded from this footnote rule, so a line ending in `p = 0.001` continues into the next line. P-value expressions without a leading zero, such as `(𝑝<.1`, also continue; a wrapped closing parenthesis attaches without an extra space. Cleanup separates paragraphs and numbered headings with an empty line, preserving existing blank lines. Other body breaks are replaced with spaces, including intervening blank lines. Single-character lines attach to the body paragraph above, without crossing numbered headings. Numbered headings such as `1. Introduction` and `2.1. Methods` remain separate and are excluded from paragraph counts. These automatic rules are heuristics; compare the resulting boundaries with the PDF.

The alternative blank-line mode is still available. Line unwrapping is optional in that mode. Word and character counts cover the displayed text, including headings and any retained mathematical text. Contractions and internally hyphenated compounds count as one word. There is no formula-exclusion or manual break-review workflow in this version.

Clipboard buttons use the local macOS clipboard. Processing runs locally without AI. Applying cleanup again replaces your edited result; copy your text before restarting the kernel or creating another result.


In [ ]:
# Run this cell if the widget cell reports that ipywidgets is missing.
# Uncomment the following line, run the cell, then restart the kernel if needed:
# %pip install ipywidgets


In [ ]:
import re
from html import escape
from unicodedata import normalize

LINE_END_HYPHEN = re.compile(r"(?<=\w)[-\u2010\u00ad][ \t]*\n[ \t]*(?=\w)")
# Numbered section/subsection headings are excluded from paragraph counts.
NUMBERED_HEADING = re.compile(r"^[^\S\n]*(?:\d+\.|\d+(?:\.\d+)+\.?)[^\S\n]+[^\W\d_].*$")
WORD = re.compile(r"[^\W_]+(?:['’\-\u2010\u2011][^\W_]+)*", re.UNICODE)


def normalize_newlines(text):
    return text.replace("\r\n", "\n").replace("\r", "\n")


def find_line_end_hyphens(text):
    """Return numbered split occurrences for the review selector."""
    text = normalize_newlines(text)
    splits = []
    for index, match in enumerate(LINE_END_HYPHEN.finditer(text)):
        left = re.search(r"[\w'’\-\u2010\u2011]+$", text[:match.start()])
        right = re.match(r"[\w'’\-\u2010\u2011]+", text[match.end():])
        left, right = left.group(0), right.group(0)
        splits.append({
            "index": index,
            "line": text.count("\n", 0, match.start()) + 1,
            "left": left,
            "right": right,
            "joined": left + right,
            "hyphenated": left + "-" + right,
        })
    return splits



def ends_paragraph(text):
    """Recognize final punctuation or footnote digits without a leading zero."""
    text = text.rstrip()
    # Normalize only for recognition: mathematical italic 𝑝 also means p.
    # A p-value can omit the leading zero, e.g. (𝑝<.1 followed by a wrap.
    if re.search(r"(?<!\w)p\s*(?:[<>=≤≥≈]\s*)+[+-]?(?:\d+)?\.\d+$",
                 normalize("NFKC", text), re.IGNORECASE):
        return False
    # A zero-prefixed suffix is likely a decimal, such as p = 0.001.
    return re.search(r"[.:?!](?:[1-9]\d*)?$", text) is not None


def attach_single_character_lines(text):
    """Attach equation remnants to the preceding body line, never across a heading."""
    output = []
    previous_body = None
    for line in normalize_newlines(text).split("\n"):
        stripped = line.strip()
        if NUMBERED_HEADING.fullmatch(line):
            output.append(line)
            previous_body = None
        elif len(stripped) == 1:
            if previous_body is not None:
                output[previous_body] = output[previous_body].rstrip() + " " + stripped
            else:
                output.append(line)
        else:
            output.append(line)
            if stripped:
                previous_body = len(output) - 1
    return "\n".join(output)


def join_unfinished_lines(text):
    """Join unfinished body lines and ensure blank lines between paragraphs."""
    parts = re.split(r"(\n(?:[^\S\n]*\n)*)", attach_single_character_lines(text))
    for index in range(1, len(parts), 2):
        previous, following = parts[index - 1], parts[index + 1]
        if (previous.strip() and following.strip()
                and not NUMBERED_HEADING.fullmatch(previous)
                and not NUMBERED_HEADING.fullmatch(following)
                and not ends_paragraph(previous)):
            parts[index - 1] = previous.rstrip()
            # Wrapped closing brackets attach directly to the expression.
            parts[index] = "" if following.lstrip().startswith((")", "]", "}")) else " "
            parts[index + 1] = following.lstrip()
        elif previous.strip() and following.strip() and parts[index].count("\n") == 1:
            # Preserve existing blank lines; add one only when it is missing.
            parts[index] = "\n\n"
    return "".join(parts)


def clean_text(text, keep_hyphens=(), unwrap_lines=False):
    """Join split words; retain a genuine hyphen for selected occurrence indexes."""
    text = normalize_newlines(text)
    keep_hyphens = set(keep_hyphens)
    matches = iter(enumerate(LINE_END_HYPHEN.finditer(text)))

    def replacement(match):
        index, _ = next(matches)
        if index not in keep_hyphens:
            return ""
        # A soft hyphen is invisible; replace it with a visible real hyphen.
        return "-" if match.group(0)[0] == "\u00ad" else match.group(0)[0]

    cleaned = join_unfinished_lines(LINE_END_HYPHEN.sub(replacement, text))
    if unwrap_lines:
        cleaned = re.sub(r"[ \t]*\n[ \t]*", "\n", cleaned)
        lines = cleaned.split("\n")
        pieces = []
        for index, line in enumerate(lines):
            pieces.append(line)
            if index + 1 < len(lines):
                following = lines[index + 1]
                join = (line.strip() and following.strip()
                        and not NUMBERED_HEADING.fullmatch(line)
                        and not NUMBERED_HEADING.fullmatch(following))
                pieces.append(" " if join else "\n")
        cleaned = "".join(pieces)
    return cleaned


def count_paragraphs(text, paragraph_mode="lines"):
    lines = join_unfinished_lines(text).split("\n")
    if paragraph_mode == "lines":
        return sum(len(line.strip()) > 1 and ends_paragraph(line)
                   and not NUMBERED_HEADING.fullmatch(line) for line in lines)
    if paragraph_mode != "blank_lines":
        raise ValueError("paragraph_mode must be 'lines' or 'blank_lines'")
    paragraphs = 0
    last_body_line = ""
    for line in lines + [""]:
        if not line.strip() or NUMBERED_HEADING.fullmatch(line):
            paragraphs += int(ends_paragraph(last_body_line))
            last_body_line = ""
        elif len(line.strip()) > 1:
            last_body_line = line
    return paragraphs


def count_text(text, paragraph_mode="lines"):
    return {
        "paragraphs": count_paragraphs(text, paragraph_mode),
        "words": len(WORD.findall(text)),
        "characters_including_whitespace": len(text),
        "characters_excluding_whitespace": sum(not char.isspace() for char in text),
    }


In [ ]:
# Checks for automatic paragraph cleanup and counting.
assert clean_text("exam-\nple.") == "example."
assert clean_text("well-\nknown.", keep_hyphens={0}) == "well-known."
assert clean_text("We define\n\nx = y\nand continue.") == "We define x = y and continue."
assert clean_text("Body.\n.\nNext.") == "Body. .\n\nNext."
assert count_text("1. Introduction\nFirst.\nSecond:")["paragraphs"] == 2
assert count_text(".")["paragraphs"] == 0
assert clean_text("Fragment\n2.1. Methods\nBody.") == "Fragment\n\n2.1. Methods\n\nBody."
# Every supported ending separates paragraphs in both counting modes.
for ending in (".", ":", "?", "!", ".1", ".12", ":2", "?3", "!45", "1."):
    original = f"First{ending}\nNext."
    expected = f"First{ending}\n\nNext."
    assert clean_text(original) == expected
    assert clean_text(original, unwrap_lines=True) == expected
    for mode in ("lines", "blank_lines"):
        assert count_text(original, paragraph_mode=mode)["paragraphs"] == 2
        assert count_text(expected, paragraph_mode=mode)["paragraphs"] == 2
    assert clean_text(expected) == expected  # Cleanup is stable when repeated.
for separator in ("\n\n", "\n\n\n", "\n \t\n"):
    original = "First?" + separator + "Next!"
    assert clean_text(original) == original
assert clean_text("First?\r\nNext!") == "First?\n\nNext!"
assert clean_text("A wrapped\nline?\nNext!") == "A wrapped line?\n\nNext!"
assert clean_text("A value of 12\ncontinues.") == "A value of 12 continues."
assert not ends_paragraph("Unfinished 12")
assert not ends_paragraph("Text. 12")  # Footnote digits must touch the punctuation.
# Zero-prefixed suffixes are decimals, not footnote paragraph endings.
for suffix in ("0", "00", "001", "01", "0001"):
    first_line = f"The result has p = 0.{suffix}"
    assert not ends_paragraph(first_line)
    for separator in ("\n", "\n\n"):
        original = first_line + separator + "and remains significant."
        expected = first_line + " and remains significant."
        assert clean_text(original) == expected
        for mode in ("lines", "blank_lines"):
            assert count_text(original, paragraph_mode=mode)["paragraphs"] == 1
# Actual sentence punctuation after a decimal still ends the paragraph.
assert clean_text("The result has p = 0.001.\nNext.") == "The result has p = 0.001.\n\nNext."
# Decimal p-values without a leading zero must not look like footnotes.
for expression in ("𝑝<.1", "p < .1", "p = .25", "𝑝 ≤ .1", "P >= .2", "p < 0.1"):
    assert not ends_paragraph("Result (" + expression)
    for separator in ("\n", "\n\n", "\r\n"):
        original = "(" + expression + separator + ")."
        expected = "(" + expression + ")."
        assert clean_text(original) == expected
        assert clean_text(original, unwrap_lines=True) == expected
        assert clean_text(expected) == expected
        for mode in ("lines", "blank_lines"):
            assert count_text(original, paragraph_mode=mode)["paragraphs"] == 1
            assert count_text(original + "\nNext.", paragraph_mode=mode)["paragraphs"] == 2
        assert clean_text(original + "\nNext.") == expected + "\n\nNext."
assert clean_text("Significant (𝑝<.1\n), with controls.") == "Significant (𝑝<.1), with controls."
assert clean_text("Result p < .1\nand other evidence.") == "Result p < .1 and other evidence."
print("Paragraph cleanup checks passed. Run the next cell to open the editor.")


In [ ]:
import shutil
import subprocess

try:
    import ipywidgets as widgets
    from IPython.display import display
except ImportError as error:
    raise ImportError("Install ipywidgets using the setup cell above, then run this cell again.") from error

original_editor = widgets.Textarea(
    description="Original text:",
    placeholder="Paste text copied from your PDF here…",
    layout=widgets.Layout(width="100%", height="240px"),
    style={"description_width": "110px"},
)
cleaned_editor = widgets.Textarea(
    description="Cleaned text:",
    placeholder="The cleaned text will appear here. You can edit it directly.",
    layout=widgets.Layout(width="100%", height="300px"),
    style={"description_width": "110px"},
)
keep_selector = widgets.SelectMultiple(
    options=[],
    description="Keep hyphens:",
    rows=6,
    layout=widgets.Layout(width="100%"),
    style={"description_width": "110px"},
)
paragraph_mode_selector = widgets.Dropdown(
    options=[("Body lines ending in . : ? or ! (optional footnote digits without leading zeros)", "lines"),
             ("Blocks separated by blank lines (wrapped text)", "blank_lines")],
    value="lines", description="Paragraph rule:",
    layout=widgets.Layout(width="100%"), style={"description_width": "110px"},
)
paragraph_rule_help = widgets.HTML()
unwrap_checkbox = widgets.Checkbox(
    value=False, disabled=True, description="Unwrap body lines (blank-line mode only)", indent=False,
    layout=widgets.Layout(width="100%"),
)
find_button = widgets.Button(description="Find line-end hyphens", layout=widgets.Layout(width="200px"))
apply_button = widgets.Button(description="Apply cleanup", button_style="primary")
clear_original_button = widgets.Button(
    description="Clear original text", button_style="danger",
    layout=widgets.Layout(width="180px", margin="0 0 0 32px", flex="0 0 auto"),
)
status = widgets.HTML()
counts_display = widgets.HTML()
headings_display = widgets.HTML()
copy_paragraph_button = widgets.Button(description="Copy paragraph count", layout=widgets.Layout(width="200px"))
copy_word_button = widgets.Button(description="Copy word count", layout=widgets.Layout(width="180px"))
copy_status = widgets.HTML()
review_state = {"source": None}


def update_counts(change=None):
    counts = count_text(cleaned_editor.value, paragraph_mode=paragraph_mode_selector.value)
    copy_status.value = ""
    counts_display.value = (
        f"<p><strong>Paragraphs:</strong> {counts['paragraphs']}<br>"
        f"<strong>Words:</strong> {counts['words']}<br>"
        f"<strong>Characters (including whitespace):</strong> "
        f"{counts['characters_including_whitespace']}<br>"
        f"<strong>Characters (excluding whitespace):</strong> "
        f"{counts['characters_excluding_whitespace']}</p>"
    )

    headings = [line.strip() for line in normalize_newlines(cleaned_editor.value).split("\n")
                if NUMBERED_HEADING.fullmatch(line)]
    headings_display.value = (
        f"<p><strong>Numbered headings excluded from paragraph count ({len(headings)}):</strong></p>"
        + "<ul>" + "".join(f"<li>{escape(heading)}</li>" for heading in headings) + "</ul>"
        if headings else ""
    )


def copy_count(metric):
    value = str(count_text(cleaned_editor.value, paragraph_mode=paragraph_mode_selector.value)[metric])
    command = shutil.which("pbcopy")
    if not command:
        copy_status.value = "<p>Copy buttons need macOS on the computer running the notebook.</p>"
        return
    try:
        subprocess.run([command], input=value, text=True, capture_output=True, check=True, timeout=5)
    except (OSError, subprocess.SubprocessError):
        copy_status.value = "<p>Could not access the clipboard. Select the displayed number and copy it manually.</p>"
        return
    label = "paragraph count" if metric == "paragraphs" else "word count"
    copy_status.value = f"<p>Copied {label}: <strong>{value}</strong></p>"


def find_splits(button=None):
    splits = find_line_end_hyphens(original_editor.value)
    keep_selector.options = [
        (f"{split['index'] + 1}. Line {split['line']}: "
         f"{split['left']}- | {split['right']} → {split['joined']} "
         f"(select to keep {split['hyphenated']})", split['index'])
        for split in splits
    ]
    keep_selector.value = ()
    review_state["source"] = original_editor.value
    status.value = (
        f"<p>Found <strong>{len(splits)}</strong> split(s). "
        "Select any that need a real hyphen, then click <strong>Apply cleanup</strong>.</p>"
    )


def apply_cleanup(button=None):
    # Selected indexes must refer to the exact original text the user reviewed.
    if review_state["source"] != original_editor.value:
        find_splits()
        if keep_selector.options:
            status.value += "<p>The original text changed. Review the refreshed list and click Apply cleanup again.</p>"
            return
    cleaned_editor.value = clean_text(
        original_editor.value, keep_hyphens=keep_selector.value,
        unwrap_lines=unwrap_checkbox.value and paragraph_mode_selector.value == "blank_lines",
    )
    update_counts()
    status.value = (
        f"<p>Cleanup applied; kept <strong>{len(keep_selector.value)}</strong> selected hyphen(s). "
        "Review and edit the cleaned text below. Counts use the current edited text.</p>"
    )


def clear_original_text(button=None):
    review_state["source"] = None
    original_editor.value = ""
    keep_selector.options = []
    keep_selector.value = ()
    status.value = "<p>Original text cleared.</p>"


def original_changed(change):
    if review_state["source"] is not None and original_editor.value != review_state["source"]:
        status.value = "<p>The original text changed. Click <strong>Find line-end hyphens</strong> to refresh the review list.</p>"


def paragraph_mode_changed(change=None):
    line_based = paragraph_mode_selector.value == "lines"
    unwrap_checkbox.disabled = line_based
    if line_based:
        unwrap_checkbox.value = False
    paragraph_rule_help.value = (
        "<p>Body lines ending in . : ? or ! (optional footnote digits without leading zeros) count as paragraphs. Cleanup adds missing blank lines between paragraphs and joins unfinished body breaks with a space. Numbered headings are excluded.</p>"
        if line_based else
        "<p>Blocks ending in . : ? or ! (optional footnote digits without leading zeros) count as paragraphs. Cleanup adds missing blank lines between paragraphs and joins breaks after unfinished body lines, including blank lines. Numbered headings are excluded.</p>"
    )
    update_counts()


copy_paragraph_button.on_click(lambda button: copy_count("paragraphs"))
copy_word_button.on_click(lambda button: copy_count("words"))
find_button.on_click(find_splits)
apply_button.on_click(apply_cleanup)
clear_original_button.on_click(clear_original_text)
original_editor.observe(original_changed, names="value")
cleaned_editor.observe(update_counts, names="value")
paragraph_mode_selector.observe(paragraph_mode_changed, names="value")
paragraph_mode_changed()
display(widgets.VBox([
    original_editor,
    widgets.HBox([
        widgets.HBox([find_button, apply_button]),
        clear_original_button,
    ], layout=widgets.Layout(width="100%", justify_content="space-between", align_items="center")),
    paragraph_mode_selector,
    paragraph_rule_help,
    unwrap_checkbox,
    status,
    keep_selector,
    widgets.HTML("<p><strong>Review the cleaned text:</strong> You can restore or remove hyphens and make any other edits here.</p>"),
    cleaned_editor,
    counts_display,
    widgets.HBox([copy_word_button, copy_paragraph_button]),
    copy_status,
    headings_display,
]))


The counts reflect **Cleaned text**. In another cell, run
`count_text(cleaned_editor.value, paragraph_mode=paragraph_mode_selector.value)`.
